# Encoder expert and routing analysis on MMEarth

This notebook evaluates the sparse **encoder** MoE of the delayed-fusion model. It uses actual top-k gates returned by the model and separates five questions that should not be conflated:

1. **Routing health:** are all experts executed and given non-trivial gate mass?
2. **Position dependence:** is expert choice tied to patch coordinates rather than content?
3. **Modality dependence:** does routing change when sensor streams change?
4. **Functional specialization:** do the expert functions and causal effects differ?
5. **Semantic specialization:** are land-cover classes enriched in particular experts?

The dense decoder has no experts and is intentionally excluded. Usage balance alone is not evidence of specialization, and visual maps are illustrations rather than statistical evidence.

## 1. Reproducible setup

Use the resolved configuration stored with the run. `deterministic=True` disables router noise and is the correct default for checkpoint comparison. The expensive semantic, ablation and trajectory sections can be disabled independently during development.

In [ ]:
from pathlib import Path
import os

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent

run_dir = Path(os.environ.get('MEOX_RUN_DIR', repo_root))
config_path = Path(os.environ.get(
    'MEOX_CONFIG', repo_root / 'configs/pretrain_mmearth_moe_mae_full.yaml'
))
checkpoint_path = Path(os.environ.get(
    'MEOX_CHECKPOINT', repo_root / 'weights/pretrained/meox_s_mmearth64_best.pth'
))
dataset_path_override = None

analysis_samples = 2048
analysis_batch_size = 64
analysis_workers = 4
analysis_seed = 42
mask_seed = 44
position_permutations = 20
deterministic = True

run_masked_analysis = True
run_semantic_analysis = True
run_expert_ablation = True
run_checkpoint_trajectory = True

inspection_dataset_index = 0
inspection_layer = 1
inspection_prefusion_modality = 'sentinel2'

In [ ]:
# Optional compact fixture for local routing checks.
use_local_test = False

if use_local_test:
    fixture_root = os.environ.get('MMEARTH_FIXTURE_ROOT')
    if not fixture_root:
        raise RuntimeError('Set MMEARTH_FIXTURE_ROOT when use_local_test=True')
    dataset_path_override = Path(fixture_root)
    analysis_samples = 128
    analysis_batch_size = 16
    analysis_workers = 0
    position_permutations = 5
    run_checkpoint_trajectory = False
    print('Local fixture mode enabled:', dataset_path_override)


In [ ]:
import math
import sys
from collections import defaultdict
from contextlib import nullcontext
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Subset

sys.path.insert(0, str(repo_root))

from pretrain_mae import _build_patch_targets, _compute_patch_loss
from utils.analysis_utils import deterministic_routing, layer_report_multimodal
from utils.extract_embeddings import (
    build_mmearth_dataset_from_config,
    build_model_from_dataset,
    build_raster_band_names,
    load_config,
    make_inference_dataloader,
    move_to_device,
    resolve_device,
    unwrap_dataset,
)

## 2. Validation data and model

The data loader reconstructs the same deterministic fallback validation split used during pretraining. The assertions prevent accidentally analyzing an old MoE-decoder checkpoint or a dataset with different input modalities.

In [ ]:
assert config_path.exists(), f'Missing resolved config: {config_path}'
assert checkpoint_path.exists(), f'Missing checkpoint: {checkpoint_path}'
config = load_config(str(config_path))
if dataset_path_override is not None:
    config['training']['dataset_path'] = str(dataset_path_override)
device = resolve_device()
dataset = build_mmearth_dataset_from_config(
    config, split='val', max_samples=analysis_samples, sample_seed=analysis_seed
)
dataset_info = unwrap_dataset(dataset)
band_names = build_raster_band_names(dataset)
dataloader = make_inference_dataloader(
    dataset, batch_size=analysis_batch_size, num_workers=analysis_workers
)
model = build_model_from_dataset(config, dataset, str(checkpoint_path), device)

assert all(not hasattr(layer, 'moe') for layer in model.decoder_layers), (
    'This notebook expects the dense-decoder architecture.'
)
assert set(dataset_info.raster_modalities) == set(model.encoder.input_specs), (
    'Dataset modalities and checkpoint input adapters differ.'
)
print(f'device={device} samples={len(dataset)} encoder_layers={len(model.encoder.layers)}')
print('modalities:', list(dataset_info.raster_modalities))
print('checkpoint:', checkpoint_path)

## 3. Routing collection

Metrics are accumulated from token counts, not averaged over batches. `active_route_fraction` is the fraction of tokens executing an expert and sums to top-k. `gate_mass_fraction` sums to one and measures the actual mixture weight. Dense local/global entropy uses the clean softmax probabilities, whereas execution metrics use the sparse gates.

The masked path below mirrors the visible-token encoder section of `MOEMAE.forward`. It is kept explicit because ordinary `forward_features` intentionally evaluates all patches. The same spatial mask is reused across sensor settings and checkpoint comparisons.

**Why this cell is slow:** for every batch it runs the 15-layer encoder once for each of seven sensor combinations, then once more with the MAE mask. With 128 local samples and batch size 16, that is 64 encoder passes on MPS/CPU. This is a real analysis run, not a lightweight plotting cell. For a quick code smoke test, use 16 samples, disable `run_masked_analysis`, and keep only `full`, `s2`, and `s1_asc` in `sensor_settings`; restore all settings for conclusions.

In [ ]:
available_modalities = tuple(dataset_info.raster_modalities)
candidate_settings = {
    'full': available_modalities,
    's2': ('sentinel2',),
    's1_asc': ('sentinel1_asc',),
    's1_desc': ('sentinel1_desc',),
    's2+s1_asc': ('sentinel2', 'sentinel1_asc'),
    's2+s1_desc': ('sentinel2', 'sentinel1_desc'),
    's1_asc+s1_desc': ('sentinel1_asc', 'sentinel1_desc'),
}
sensor_settings = {
    name: modalities for name, modalities in candidate_settings.items()
    if modalities and set(modalities).issubset(available_modalities)
}

def routing_views(features):
    for modality, routing in features['routing'][0]['by_modality'].items():
        yield 0, modality, routing
    fine_start = int(features['token_layout']['num_meta_tokens']) + 1
    fine_count = int(features['token_layout']['num_fine_tokens'])
    fine_end = fine_start + fine_count
    for layer_index, routing in enumerate(features['routing'][1:], start=1):
        yield layer_index, 'fused', {
            key: value[:, fine_start:fine_end]
            if torch.is_tensor(value) and value.ndim >= 3 else value
            for key, value in routing.items() if key != 'stage'
        }

def masked_forward_features(
    active_model, rasters, validity, raster_bands, metadata, metadata_validity, ids_keep
):
    encoder = active_model.encoder
    tokens, patch_validity, layout, _ = encoder._prepare_modality_tokens(
        raster_dict=rasters, raster_valid_masks=validity, raster_band_names=raster_bands
    )
    first = next(iter(tokens.values()))
    batch_size, _, dim = first.shape
    grid_y, grid_x = torch.meshgrid(
        torch.arange(layout['fine_height'], device=first.device),
        torch.arange(layout['fine_width'], device=first.device),
        indexing='ij',
    )
    positions = torch.stack([grid_x, grid_y], dim=-1).reshape(1, -1, 2)
    positions = positions.float().expand(batch_size, -1, -1)
    visible_positions = positions.gather(
        1, ids_keep.unsqueeze(-1).expand(-1, -1, 2)
    )
    visible_tokens = {
        name: value.gather(1, ids_keep.unsqueeze(-1).expand(-1, -1, dim))
        for name, value in tokens.items()
    }
    visible_validity = {
        name: value.gather(1, ids_keep.unsqueeze(-1))
        for name, value in patch_validity.items()
    }
    meta_tokens = encoder._build_metadata(
        batch_size, first.device, first.dtype,
        meta_dict=metadata, meta_valid_masks=metadata_validity,
    )
    encoded = encoder._encode_modalities(
        visible_tokens, visible_validity, meta_tokens, layout, visible_positions,
        return_routing=True, stochastic_routing=not deterministic,
    )
    return {
        'token_layout': encoded[3],
        'fusion_weights': encoded[4],
        'fusion_modalities': encoded[5],
        'routing': encoded[6],
    }

def fixed_visible_indices(batch_size, num_patches, batch_index):
    generator = torch.Generator().manual_seed(mask_seed + batch_index)
    noise = torch.rand(batch_size, num_patches, generator=generator)
    keep_count = int(num_patches * (1.0 - model.mask_ratio))
    return noise.argsort(dim=1)[:, :keep_count].to(device)

In [ ]:
def new_accumulator(num_experts):
    return {
        'tokens': 0,
        'routes': torch.zeros(num_experts, dtype=torch.float64),
        'gate_mass': torch.zeros(num_experts, dtype=torch.float64),
        'top1': torch.zeros(num_experts, dtype=torch.float64),
        'probability': torch.zeros(num_experts, dtype=torch.float64),
        'local_entropy_sum': 0.0,
        'balance_loss_sum': 0.0,
    }

def update_accumulator(accumulators, key, routing):
    gates = routing['gates'].detach().float().reshape(-1, routing['gates'].shape[-1]).cpu()
    probabilities = routing['clean_logits'].detach().float().softmax(dim=-1)
    probabilities = probabilities.reshape(-1, probabilities.shape[-1]).cpu()
    if key not in accumulators:
        accumulators[key] = new_accumulator(gates.shape[-1])
    acc = accumulators[key]
    token_count = gates.shape[0]
    acc['tokens'] += token_count
    acc['routes'] += (gates > 0).sum(dim=0)
    acc['gate_mass'] += gates.sum(dim=0)
    acc['top1'] += F.one_hot(gates.argmax(dim=-1), gates.shape[-1]).sum(dim=0)
    acc['probability'] += probabilities.sum(dim=0)
    acc['local_entropy_sum'] += float(
        (-(probabilities.clamp_min(1e-9).log() * probabilities).sum(dim=-1)).sum()
    )
    acc['balance_loss_sum'] += float(routing['balance_loss']) * token_count

def finalize_accumulators(accumulators):
    rows = []
    for (mode, setting, layer, stream), acc in accumulators.items():
        count = acc['tokens']
        probability = acc['probability'] / count
        num_experts = len(probability)
        local_entropy = acc['local_entropy_sum'] / count / math.log(num_experts)
        global_entropy = float(
            -(probability.clamp_min(1e-9).log() * probability).sum() / math.log(num_experts)
        )
        for expert in range(num_experts):
            rows.append({
                'mode': mode, 'setting': setting, 'layer': layer, 'stream': stream,
                'expert': expert, 'tokens': count,
                'active_route_fraction': float(acc['routes'][expert] / count),
                'gate_mass_fraction': float(acc['gate_mass'][expert] / count),
                'top1_fraction': float(acc['top1'][expert] / count),
                'dense_probability': float(probability[expert]),
                'local_entropy': local_entropy, 'global_entropy': global_entropy,
                'balance_loss': acc['balance_loss_sum'] / count,
            })
    return pd.DataFrame(rows)

def full_position_map(routing, total_patches, ids_keep=None):
    assignment = routing['gates'].argmax(dim=-1).detach().cpu().numpy().astype(np.int16)
    if ids_keep is None:
        return assignment
    output = np.full((assignment.shape[0], total_patches), -1, dtype=np.int16)
    np.put_along_axis(output, ids_keep.detach().cpu().numpy(), assignment, axis=1)
    return output

routing_accumulators = {}
position_blocks = defaultdict(list)
agreement_accumulators = defaultdict(lambda: {'top1_same': 0, 'jaccard_sum': 0.0, 'tokens': 0})
fusion_blocks = defaultdict(list)
fusion_tile_blocks = defaultdict(list)
fusion_within_tile_blocks = defaultdict(list)
analysis_tile_ids = []

context = deterministic_routing(model) if deterministic else nullcontext()
model.eval()
with torch.inference_mode(), context:
    for batch_index, batch in enumerate(dataloader):
        analysis_tile_ids.extend([str(tile_id) for tile_id in batch['tile_id']])
        full_rasters = move_to_device(batch['raster_dict'], device)
        full_validity = move_to_device(batch.get('raster_valid_masks'), device)
        metadata = move_to_device(batch.get('meta_dict'), device)
        metadata_validity = move_to_device(batch.get('meta_valid_masks'), device)
        setting_features = {}
        for setting, modalities in sensor_settings.items():
            rasters = {name: full_rasters[name] for name in modalities}
            validity = ({name: full_validity[name] for name in modalities}
                        if full_validity is not None else None)
            features = model.forward_features(
                raster_dict=rasters, raster_valid_masks=validity,
                raster_band_names={name: band_names[name] for name in modalities},
                meta_dict=metadata, meta_valid_masks=metadata_validity,
                return_routing=True, stochastic_routing=not deterministic,
            )
            setting_features[setting] = features
            for layer, stream, routing in routing_views(features):
                update_accumulator(
                    routing_accumulators, ('full', setting, layer, stream), routing
                )
                if setting == 'full':
                    position_blocks[('full', layer, stream)].append(
                        full_position_map(routing, features['token_layout']['num_fine_tokens'])
                    )
        full_features = setting_features['full']
        fusion = full_features['fusion_weights'].detach().float().cpu().numpy()
        for modality_index, modality in enumerate(full_features['fusion_modalities']):
            fusion_blocks[modality].append(fusion[..., modality_index].reshape(-1))
            modality_weights = fusion[..., modality_index]
            fusion_tile_blocks[modality].append(modality_weights.mean(axis=1))
            fusion_within_tile_blocks[modality].append(modality_weights.std(axis=1))

        full_routes = {layer: routing for layer, stream, routing in routing_views(full_features)
                       if stream == 'fused'}
        for setting, features in setting_features.items():
            if setting == 'full':
                continue
            for layer, stream, routing in routing_views(features):
                if stream != 'fused':
                    continue
                reference = full_routes[layer]['gates']
                candidate = routing['gates']
                ref_active, cand_active = reference > 0, candidate > 0
                token_count = reference.shape[0] * reference.shape[1]
                acc = agreement_accumulators[(setting, layer)]
                acc['top1_same'] += int((reference.argmax(-1) == candidate.argmax(-1)).sum())
                intersection = (ref_active & cand_active).sum(-1).float()
                union = (ref_active | cand_active).sum(-1).float().clamp_min(1)
                acc['jaccard_sum'] += float((intersection / union).sum())
                acc['tokens'] += token_count

        if run_masked_analysis:
            sample_raster = next(iter(full_rasters.values()))
            patch_rows = sample_raster.shape[-2] // model.encoder.patch_size
            patch_columns = sample_raster.shape[-1] // model.encoder.patch_size
            num_patches = patch_rows * patch_columns
            ids_keep = fixed_visible_indices(len(batch['tile_id']), num_patches, batch_index)
            masked = masked_forward_features(
                model, full_rasters, full_validity, band_names, metadata, metadata_validity, ids_keep
            )
            for layer, stream, routing in routing_views(masked):
                update_accumulator(
                    routing_accumulators, ('masked', 'full', layer, stream), routing
                )
                position_blocks[('masked', layer, stream)].append(
                    full_position_map(routing, num_patches, ids_keep=ids_keep)
                )

routing_summary = finalize_accumulators(routing_accumulators)
position_assignments = {key: np.concatenate(blocks, axis=0)
                        for key, blocks in position_blocks.items()}
routing_summary.head()

## 4. Routing health

Inspect sparse execution and gate mass together. A low active fraction means an expert rarely runs; a low mass means it contributes little even when selected. Local entropy measures token-level uncertainty, while global entropy only measures the average dense distribution and therefore cannot establish balanced sparse routing by itself.

In [ ]:
health = routing_summary.query("setting == 'full'").groupby(
    ['mode', 'layer', 'stream'], as_index=False
).agg(
    min_active_route=('active_route_fraction', 'min'),
    min_gate_mass=('gate_mass_fraction', 'min'),
    min_top1=('top1_fraction', 'min'),
    local_entropy=('local_entropy', 'first'),
    global_entropy=('global_entropy', 'first'),
    balance_loss=('balance_loss', 'first'),
)
display(health.round(4))

for metric, title in [
    ('active_route_fraction', 'Executed top-k route fraction'),
    ('gate_mass_fraction', 'Executed gate mass'),
]:
    figure, axes = plt.subplots(1, 2 if run_masked_analysis else 1, figsize=(13, 4), squeeze=False)
    for axis, mode in zip(axes[0], ['full', 'masked'] if run_masked_analysis else ['full']):
        values = routing_summary.query(
            "setting == 'full' and stream == 'fused' and mode == @mode"
        ).pivot(index='layer', columns='expert', values=metric)
        image = axis.imshow(values, aspect='auto', cmap='viridis', vmin=0)
        axis.set(title=f'{mode}: {title}', xlabel='Expert', ylabel='Layer')
        axis.set_xticks(range(len(values.columns)), values.columns)
        axis.set_yticks(range(len(values.index)), values.index)
        figure.colorbar(image, ax=axis)
    plt.tight_layout()
    plt.show()

### Interpretation of the current local run

**What the test does.** For every layer and expert, it counts how many tokens include that expert in top-2 (`active_route_fraction`), sums its executed mixture weights (`gate_mass_fraction`), and counts how often it is top-1. Active fractions sum to 2 because each token executes two experts; gate mass and top-1 fractions each sum to 1. The uniform references are therefore `2 / E` for active routes and `1 / E` for gate mass/top-1, where `E` is the number of experts in that layer. A Switch-style balance loss near 1 is ideal.

**Result.** There is no dead or globally starved encoder expert. Across fused layers, the minimum full-input active fraction is 0.251 and the minimum gate mass is 0.116; under the 75% MAE mask those minima improve to 0.341 and 0.163 in the deep layers. Balance losses remain between about 1.00 and 1.07. Layer 1 and full-input layer 14 are less balanced than the others, but this is skew rather than collapse.

At layer 0, ascending S1 strongly favors expert 1: that expert is selected for almost every patch and receives 52.9% of gate mass. S2 is nearly uniform across its three experts. Because the same layer is applied separately to all modalities and every expert remains used across the complete input, this indicates modality-conditioned routing, not a dead expert. The full and masked heatmaps should both be retained: masked routing is closest to the pretraining regime, while full routing is closest to ordinary downstream inference.

## 5. Position-dependence test

Normalized mutual information (NMI) measures dependence between absolute patch position and top-1 expert. Its permutation distribution is the finite-sample null. Same-position agreement is compared with the agreement expected from global expert frequencies. Report **excess over the null/chance**, not raw NMI or agreement alone.

In [ ]:
def contingency_nmi(table):
    table = np.asarray(table, dtype=np.float64)
    total = table.sum()
    if total == 0:
        return np.nan
    joint = table / total
    row = joint.sum(axis=1, keepdims=True)
    col = joint.sum(axis=0, keepdims=True)
    expected = row @ col
    valid = joint > 0
    mutual_information = np.sum(joint[valid] * np.log(joint[valid] / expected[valid]))
    row_entropy = -np.sum(row[row > 0] * np.log(row[row > 0]))
    col_entropy = -np.sum(col[col > 0] * np.log(col[col > 0]))
    denominator = math.sqrt(row_entropy * col_entropy)
    return mutual_information / denominator if denominator > 0 else 0.0

def position_report(assignments, permutations, seed):
    batch_size, num_positions = assignments.shape
    position_grid = np.broadcast_to(np.arange(num_positions), assignments.shape)
    valid = assignments >= 0
    positions = position_grid[valid]
    experts = assignments[valid]
    num_experts = int(experts.max()) + 1
    table = np.zeros((num_positions, num_experts), dtype=np.int64)
    np.add.at(table, (positions, experts), 1)
    observed_nmi = contingency_nmi(table)
    rng = np.random.default_rng(seed)
    null_nmi = []
    for _ in range(permutations):
        shuffled = rng.permutation(experts)
        null_table = np.zeros_like(table)
        np.add.at(null_table, (positions, shuffled), 1)
        null_nmi.append(contingency_nmi(null_table))
    position_pairs = table.sum(axis=1) * (table.sum(axis=1) - 1)
    same_pairs = (table * (table - 1)).sum(axis=1)
    same_position = same_pairs.sum() / max(position_pairs.sum(), 1)
    global_counts = table.sum(axis=0)
    chance = (global_counts * (global_counts - 1)).sum() / max(
        global_counts.sum() * (global_counts.sum() - 1), 1
    )
    return {
        'position_nmi': observed_nmi,
        'null_nmi_mean': float(np.mean(null_nmi)),
        'null_nmi_95': float(np.quantile(null_nmi, 0.95)),
        'nmi_excess': observed_nmi - float(np.mean(null_nmi)),
        'same_position_agreement': same_position,
        'marginal_chance': chance,
        'agreement_excess': same_position - chance,
    }

position_rows = []
for index, ((mode, layer, stream), assignments) in enumerate(position_assignments.items()):
    position_rows.append({
        'mode': mode, 'layer': layer, 'stream': stream,
        **position_report(assignments, position_permutations, analysis_seed + index),
    })
position_summary = pd.DataFrame(position_rows).sort_values(['mode', 'layer', 'stream'])
display(position_summary.round(5))

for metric in ['nmi_excess', 'agreement_excess']:
    for mode, values in position_summary.query("stream == 'fused'").groupby('mode'):
        plt.plot(values['layer'], values[metric], marker='o', label=mode)
    plt.axhline(0, color='black', linewidth=1)
    plt.title(metric.replace('_', ' ').title())
    plt.xlabel('Layer')
    plt.legend()
    plt.show()

### Interpretation of the current local run

**What the test does.** It builds a contingency table between absolute patch index and top-1 expert. NMI is zero when these variables are independent. The permutation null repeatedly shuffles expert labels while preserving their global frequencies; `nmi_excess` subtracts the mean null NMI. The second statistic asks whether two different tiles at the same patch coordinate choose the same expert more often than expected from marginal expert frequencies.

**Result.** There is no practically meaningful absolute-position routing pattern. For full inputs, NMI excess ranges from approximately -0.0024 to +0.0007 and agreement excess from -0.0037 to +0.0012. Masked values are similarly close to zero. Full-input layer 14 is slightly above the small five-permutation local null, but the effect size is only 0.00066 NMI and 0.00121 agreement; that is not evidence of the fixed spatial routing previously observed.

The local override uses only five permutations, which is enough to catch a large shortcut but not to report a stable p-value. For final reporting, use at least 100 permutations and confidence intervals across tiles or seeds. A patch-level permutation is a descriptive null because neighboring patches within one tile are correlated.

## 6. Modality routing and sensor removal

Layer 0 processes each sensor independently with shared weights, so its rows directly compare modality-conditioned routing. Later layers receive fused tokens. For sensor removal, total-variation (TV) distance compares aggregate gate-mass distributions, top-1 agreement compares the selected primary expert token-by-token, and top-k Jaccard compares the executed expert sets. Entropy alone is deliberately not used as a change metric.

In [ ]:
prefusion = routing_summary.query("mode == 'full' and setting == 'full' and layer == 0")
prefusion_mass = prefusion.pivot(index='stream', columns='expert', values='gate_mass_fraction')
display(prefusion_mass.round(4))
plt.figure(figsize=(7, 3))
image = plt.imshow(prefusion_mass, aspect='auto', cmap='viridis', vmin=0)
plt.colorbar(image, label='Gate mass')
plt.xticks(range(len(prefusion_mass.columns)), prefusion_mass.columns)
plt.yticks(range(len(prefusion_mass.index)), prefusion_mass.index)
plt.title('Layer 0 routing conditioned on sensor')
plt.xlabel('Expert')
plt.show()

removal_rows = []
for (setting, layer), acc in agreement_accumulators.items():
    full_mass = routing_summary.query(
        "mode == 'full' and setting == 'full' and stream == 'fused' and layer == @layer"
    ).sort_values('expert')['gate_mass_fraction'].to_numpy()
    changed_mass = routing_summary.query(
        "mode == 'full' and setting == @setting and stream == 'fused' and layer == @layer"
    ).sort_values('expert')['gate_mass_fraction'].to_numpy()
    removal_rows.append({
        'setting': setting, 'layer': layer,
        'gate_mass_tv': 0.5 * np.abs(full_mass - changed_mass).sum(),
        'top1_agreement': acc['top1_same'] / acc['tokens'],
        'topk_jaccard': acc['jaccard_sum'] / acc['tokens'],
    })
removal_summary = pd.DataFrame(removal_rows)
display(removal_summary.round(4))

figure, axes = plt.subplots(1, 3, figsize=(17, 4))
for axis, metric in zip(axes, ['gate_mass_tv', 'top1_agreement', 'topk_jaccard']):
    for setting, values in removal_summary.groupby('setting'):
        axis.plot(values['layer'], values[metric], marker='o', label=setting)
    axis.set(title=metric.replace('_', ' ').title(), xlabel='Layer')
axes[-1].legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

### Interpretation of the current local run

**What the test does.** Layer 0 compares routing for S2, ascending S1, and descending S1 before fusion, using one shared MoE layer. The removal test then reruns the same samples with selected sensors omitted. Gate-mass total variation (TV) measures aggregate distribution change: 0 means identical and 1 means disjoint. Top-1 agreement compares the primary expert per patch, while top-k Jaccard compares the two executed expert sets.

**Result.** Layer 0 is clearly modality-conditioned. S2 gate mass is nearly balanced (0.324, 0.330, 0.346), while ascending S1 gives expert 1 a mass of 0.529; descending S1 also favors expert 1, but less strongly (0.420). This supports sensor-dependent routing through shared experts, not one exclusive expert per sensor.

Removing sensors changes deeper routing. S2-only input differs most strongly from the full model at layer 1 (TV 0.502, top-1 agreement 0.262), which is expected because this is the first layer after multimodal fusion. Later differences are smaller but persistent. Two-sensor combinations generally preserve routes better than a single sensor. These results show that routing responds to available sensor content; they do not by themselves establish modality-specialized experts.

## 7. Fusion statistics and missing-sensor check

Fusion weights describe sensor contribution before the fused encoder layers. Report their distribution and both within-tile and between-tile variability; a global mean alone can hide constant or spatially varying behavior. The controlled validity test verifies that a fully invalid sensor receives negligible weight.

In [ ]:
fusion_rows = []
for modality, blocks in fusion_blocks.items():
    values = np.concatenate(blocks)
    tile_means = np.concatenate(fusion_tile_blocks[modality])
    within_tile = np.concatenate(fusion_within_tile_blocks[modality])
    fusion_rows.append({
        'modality': modality, 'mean': values.mean(), 'std': values.std(),
        'p05': np.quantile(values, 0.05), 'median': np.median(values),
        'p95': np.quantile(values, 0.95),
        'mean_within_tile_std': within_tile.mean(),
        'between_tile_std': tile_means.std(),
    })
fusion_summary = pd.DataFrame(fusion_rows)
display(fusion_summary.round(4))

check_batch = next(iter(make_inference_dataloader(
    Subset(dataset, [inspection_dataset_index]), batch_size=1, num_workers=0
)))
check_rasters = move_to_device(check_batch['raster_dict'], device)
check_validity = move_to_device(check_batch['raster_valid_masks'], device)
missing_rows = []
with torch.inference_mode(), deterministic_routing(model):
    for missing_modality in available_modalities:
        controlled_validity = {name: value.clone() for name, value in check_validity.items()}
        controlled_validity[missing_modality].zero_()
        features = model.forward_features(
            raster_dict=check_rasters, raster_valid_masks=controlled_validity,
            raster_band_names=band_names,
            meta_dict=move_to_device(check_batch.get('meta_dict'), device),
            meta_valid_masks=move_to_device(check_batch.get('meta_valid_masks'), device),
        )
        modality_index = features['fusion_modalities'].index(missing_modality)
        missing_rows.append({
            'missing_modality': missing_modality,
            'maximum_fusion_weight': float(features['fusion_weights'][..., modality_index].max()),
        })
display(pd.DataFrame(missing_rows))

### Interpretation of the current local run

**What the test does.** It summarizes every learned per-patch fusion weight, separates spatial variation within a tile from variation between tile means, and then sets each sensor validity mask to zero in turn. The invalid-sensor test checks a hard correctness property: unavailable data should receive essentially zero fusion weight.

**Result.** Fusion does not collapse onto one sensor. Mean weights are 0.250 for S2, 0.378 for ascending S1, and 0.371 for descending S1. The invalid-sensor maxima are only 4.4e-7 to 1.5e-6, so validity masking works correctly.

The learned weights vary only modestly: mean within-tile standard deviations are 0.008-0.014 and between-tile standard deviations are about 0.013. Thus, in this fixture, fusion behaves mainly like a learned sensor prior with small content-dependent adjustments rather than aggressively changing sensor weight patch by patch. This is healthy but weaker than a claim that fusion automatically identifies clouds or local sensor corruption; such a claim requires conditioning weights on cloud masks, invalid pixels, or controlled corruptions.

## 8. Semantic specialization with Dynamic World

The label dataset uses exactly the same global validation indices as the model-input dataset. Dynamic World pixels are reduced to patch-majority labels using their validity mask. Expert/class NMI is calibrated by label permutation. Class enrichment heatmaps show `P(class | expert) / P(class)`; values above one indicate enrichment, not necessarily causal specialization.

In [ ]:
def resolved_global_indices(active_dataset):
    if isinstance(active_dataset, Subset):
        parent = resolved_global_indices(active_dataset.dataset)
        return [parent[index] for index in active_dataset.indices]
    return list(active_dataset.indices)

def patch_majority(labels, validity, patch_size, num_classes=9):
    label_patches = F.unfold(labels.float(), patch_size, stride=patch_size).transpose(1, 2)
    valid_patches = F.unfold(validity.float(), patch_size, stride=patch_size).transpose(1, 2) > 0
    counts = torch.stack([
        ((label_patches == class_index) & valid_patches).sum(dim=-1)
        for class_index in range(num_classes)
    ], dim=-1)
    majority = counts.argmax(dim=-1)
    majority[counts.sum(dim=-1) == 0] = -1
    return majority

semantic_summary = pd.DataFrame()
semantic_lifts = {}
if run_semantic_analysis:
    selected_indices = resolved_global_indices(dataset)
    label_dataset = build_mmearth_dataset_from_config(
        config, split='val', raster_modalities=['dynamic_world'], metadata_modalities=[],
        explicit_indices=selected_indices,
    )
    label_loader = make_inference_dataloader(
        label_dataset, batch_size=analysis_batch_size, num_workers=analysis_workers
    )
    label_blocks = []
    label_tiles = []
    for label_batch in label_loader:
        label_blocks.append(patch_majority(
            label_batch['raster_dict']['dynamic_world'],
            label_batch['raster_valid_masks']['dynamic_world'],
            model.encoder.patch_size,
        ).numpy())
        label_tiles.extend([str(tile_id) for tile_id in label_batch['tile_id']])
    labels = np.concatenate(label_blocks, axis=0)
    assert label_tiles == analysis_tile_ids, 'Dynamic World labels are not aligned to routing samples.'

    semantic_rows = []
    rng = np.random.default_rng(analysis_seed)
    for (mode, layer, stream), assignments in position_assignments.items():
        if mode != 'full':
            continue
        valid = (labels >= 0) & (assignments >= 0)
        expert_values = assignments[valid]
        class_values = labels[valid]
        num_experts = int(expert_values.max()) + 1
        table = np.zeros((num_experts, 9), dtype=np.int64)
        np.add.at(table, (expert_values, class_values), 1)
        observed = contingency_nmi(table)
        null = []
        for _ in range(position_permutations):
            shuffled = rng.permutation(class_values)
            null_table = np.zeros_like(table)
            np.add.at(null_table, (expert_values, shuffled), 1)
            null.append(contingency_nmi(null_table))
        class_prior = table.sum(axis=0) / table.sum()
        conditional = table / np.maximum(table.sum(axis=1, keepdims=True), 1)
        semantic_lifts[(layer, stream)] = conditional / np.maximum(class_prior, 1e-12)
        semantic_rows.append({
            'layer': layer, 'stream': stream, 'expert_class_nmi': observed,
            'null_nmi_mean': np.mean(null), 'null_nmi_95': np.quantile(null, 0.95),
            'nmi_excess': observed - np.mean(null),
        })
    semantic_summary = pd.DataFrame(semantic_rows).sort_values(['layer', 'stream'])
    display(semantic_summary.round(5))

    selected_layers = [0, len(model.encoder.layers) // 2, len(model.encoder.layers) - 1]
    selected_keys = []
    for layer in selected_layers:
        candidates = [key for key in semantic_lifts if key[0] == layer]
        if layer == 0:
            candidates = [key for key in candidates if key[1] == 'sentinel2'] or candidates[:1]
        else:
            candidates = [key for key in candidates if key[1] == 'fused']
        selected_keys.extend(candidates[:1])
    figure, axes = plt.subplots(1, len(selected_keys), figsize=(6 * len(selected_keys), 3), squeeze=False)
    for axis, key in zip(axes[0], selected_keys):
        image = axis.imshow(semantic_lifts[key], aspect='auto', cmap='coolwarm', vmin=0, vmax=2)
        axis.set(title=f'Layer {key[0]} ({key[1]}) class lift', xlabel='Dynamic World class', ylabel='Expert')
        figure.colorbar(image, ax=axis)
    plt.tight_layout()
    plt.show()
else:
    print('Semantic analysis disabled.')

### Interpretation of the current local run

**What the test does.** Dynamic World pixels are remapped to classes 0-8 (`water`, `trees`, `grass`, `flooded vegetation`, `crops`, `shrub/scrub`, `built`, `bare`, `snow/ice`) and reduced to one majority label per model patch. Expert/class NMI measures association between top-1 routing and land cover. The heatmap reports class lift, `P(class | expert) / P(class)`: 1 is no enrichment, above 1 is enrichment, and below 1 is depletion.

**Result.** Routing contains clear land-cover information at every layer. Observed expert/class NMI ranges from 0.027 to 0.123, while the local permutation null is only about 0.0001-0.0004. The strongest association occurs around layers 2 and 5, and it remains non-zero at layer 14. The heatmaps also show different profiles rather than one shared class distribution; for example, the S2 layer-0 experts differ in enrichment for water/flooded vegetation versus built/bare classes.

This supports semantic association, not pure semantic experts. An expert can respond to several classes, and NMI does not prove causality. Because patches from one tile are spatially correlated, the patch-level permutation overstates the number of independent observations. For publication, repeat across seeds and checkpoints and use tile-level bootstrap intervals or a block/tile-aware permutation.

## 9. Qualitative true-routing maps

Select a dataset index explicitly; the loader is deterministic, so repeatedly calling `next(iter(dataloader))` intentionally returns the same first batch. Layer-0 S1 routes are shown over a VV/VH pseudo-RGB background rather than an unrelated S2 image. Later fused layers use S2 RGB for geographic context.

In [ ]:
def percentile_rgb(channels):
    image = channels.permute(1, 2, 0).float().numpy()
    low, high = np.percentile(image, (2, 98), axis=(0, 1), keepdims=True)
    return np.clip((image - low) / np.maximum(high - low, 1e-6), 0, 1)

def sentinel2_rgb(tensor, bands):
    return percentile_rgb(tensor[[bands.index(name) for name in ('B4', 'B3', 'B2')]])

def sentinel1_rgb(tensor, bands):
    vv = tensor[bands.index('VV')]
    vh = tensor[bands.index('VH')]
    return percentile_rgb(torch.stack([vv, vh, 0.5 * (vv + vh)]))

inspection_loader = make_inference_dataloader(
    Subset(dataset, [inspection_dataset_index]), batch_size=1, num_workers=0
)
inspection_batch = next(iter(inspection_loader))
if inspection_layer == 0 and inspection_prefusion_modality.startswith('sentinel1'):
    base_rgb = sentinel1_rgb(
        inspection_batch['raster_dict'][inspection_prefusion_modality][0],
        band_names[inspection_prefusion_modality],
    )
    background = f'{inspection_prefusion_modality} VV/VH pseudo-RGB'
else:
    base_rgb = sentinel2_rgb(
        inspection_batch['raster_dict']['sentinel2'][0], band_names['sentinel2']
    )
    background = 'Sentinel-2 RGB'
report = layer_report_multimodal(
    model=model,
    raster_dict=move_to_device(inspection_batch['raster_dict'], device),
    raster_valid_masks=move_to_device(inspection_batch.get('raster_valid_masks'), device),
    raster_band_names=band_names, base_rgb=base_rgb,
    meta_dict=move_to_device(inspection_batch.get('meta_dict'), device),
    meta_valid_masks=move_to_device(inspection_batch.get('meta_valid_masks'), device),
    image_index=0, layer_index=inspection_layer,
    prefusion_modality=inspection_prefusion_modality, deterministic=deterministic,
)
print('dataset index:', inspection_dataset_index)
print('tile:', inspection_batch['tile_id'][0])
print('background:', background)
print('active routes:', report['usage_image'].tolist())

### Interpretation of the current local example

**What the test shows.** Colored dots are the top-1 expert at each patch. The second figure shows the actual sparse top-2 gate weights; black cells have zero weight because that expert was not selected. `active routes` counts top-2 execution, not top-1 assignment, so counts sum to twice the number of patches.

**Result.** This 16 x 16 example has route counts `[232, 27, 253]`, which sum to 512 as required for 256 patches with top-2 routing. Expert 1 is rarely selected for this particular tile, whereas experts 0 and 2 divide most locations with spatially varying weights. That is compatible with the global layer-1 active fraction of at least 0.313: a locally sparse expert is not a globally collapsed expert.

Use the map to generate hypotheses, not conclusions. Change `inspection_dataset_index`, inspect several land-cover types, and compare layer 0 separately for S2 and both S1 streams. Repeated identical maps across many unrelated samples would be suspicious; one imbalanced sample is normal for content-dependent routing.

## 10. Expert-function similarity

Routing diversity does not guarantee different expert functions. A pre-hook captures the normalized inputs actually sent to each MoE. Every expert is then evaluated on the same token sample. Mean output norms detect inactive functions, while pairwise output cosine similarity detects experts that compute nearly the same direction. Shared value/output projections make some similarity expected; the expert-specific gate and low-rank adapters should still produce measurable differences.

In [ ]:
def expert_function_report(active_model, batch, max_tokens=4096):
    captured = defaultdict(list)
    hooks = []
    for layer_index, layer in enumerate(active_model.encoder.layers):
        hooks.append(layer.moe.register_forward_pre_hook(
            lambda module, inputs, index=layer_index: captured[index].append(inputs[0].detach())
        ))
    try:
        with torch.inference_mode(), deterministic_routing(active_model):
            active_model.forward_features(
                raster_dict=move_to_device(batch['raster_dict'], device),
                raster_valid_masks=move_to_device(batch.get('raster_valid_masks'), device),
                raster_band_names=band_names,
                meta_dict=move_to_device(batch.get('meta_dict'), device),
                meta_valid_masks=move_to_device(batch.get('meta_valid_masks'), device),
            )
    finally:
        for hook in hooks:
            hook.remove()

    norm_rows, cosine_rows = [], []
    with torch.inference_mode():
        for layer_index, layer in enumerate(active_model.encoder.layers):
            tokens = torch.cat(captured[layer_index], dim=0).reshape(-1, active_model.encoder.embed_dim)
            if len(tokens) > max_tokens:
                indices = torch.linspace(0, len(tokens) - 1, max_tokens, device=tokens.device).long()
                tokens = tokens[indices]
            outputs = [expert(tokens).float() for expert in layer.moe.experts]
            for expert_index, output in enumerate(outputs):
                norm_rows.append({
                    'layer': layer_index, 'expert': expert_index,
                    'mean_output_norm': float(output.norm(dim=-1).mean()),
                })
            for first, second in combinations(range(len(outputs)), 2):
                cosine_rows.append({
                    'layer': layer_index, 'pair': f'E{first}-E{second}',
                    'mean_output_cosine': float(F.cosine_similarity(
                        outputs[first], outputs[second], dim=-1
                    ).mean()),
                })
    return pd.DataFrame(norm_rows), pd.DataFrame(cosine_rows)

function_sample_count = min(32, len(dataset))
function_batch = next(iter(make_inference_dataloader(
    Subset(dataset, list(range(function_sample_count))),
    batch_size=function_sample_count, num_workers=0,
)))
function_norms, function_cosines = expert_function_report(model, function_batch)
display(function_norms.pivot(index='layer', columns='expert', values='mean_output_norm').round(3))
display(function_cosines.pivot(index='layer', columns='pair', values='mean_output_cosine').round(3))
function_cosines.pivot(index='layer', columns='pair', values='mean_output_cosine').plot(
    marker='o', figsize=(10, 4), title='Expert-output cosine on shared token inputs'
)
plt.ylabel('Mean cosine similarity')
plt.show()

### Interpretation of the current local run

**What the test does.** A hook captures the normalized input to each MoE layer. Every expert in that layer is then evaluated on exactly the same tokens, removing routing selection as a confound. Output norm checks whether an expert produces a non-trivial function; pairwise cosine compares output direction, where 1 means functionally parallel on these inputs and lower values mean greater differentiation.

**Result.** No expert is functionally dead: every reported output norm is non-zero. Early experts are relatively similar (pairwise cosine about 0.68-0.89), which is expected because experts share value and output projections. Similarity decreases with depth; around layer 12 it is approximately 0.39-0.47, showing substantial expert-specific divergence. Layer 14 rises to moderate similarity again but remains far from identical.

The `NaN` cells are expected and do not indicate broken experts. The architecture uses three experts in early layers, four in middle layers, and five in deep layers, so impossible expert pairs are absent from the pivot table. Layer-14 expert 0 has a larger output norm (1.91 versus roughly 1.08-1.23), which should be read together with its stronger ablation effect below. Functional difference is necessary for specialization, but it does not establish what each expert represents.

## 11. Causal expert ablation

This experiment suppresses one executed expert output at a time while leaving routing unchanged. The same deterministic MAE mask is reused for every run. Reconstruction-loss increase measures pretraining-task contribution; embedding cosine distance measures representation change. Layer 0 uses the same shared expert for each sensor stream, so its ablation affects that expert in all pre-fusion calls. Large impact is contribution, not automatically semantic specialization.

In [ ]:
def fixed_mask_generator(seed):
    return torch.Generator().manual_seed(seed)

def reconstruction_by_modality(predictions, targets, validity, mask):
    losses = {}
    for modality, target in targets.items():
        weights = mask.unsqueeze(-1) * validity[modality]
        losses[modality] = float(
            (((predictions[modality] - target) ** 2) * weights).sum()
            / weights.sum().clamp_min(1)
        )
    return losses

ablation_summary = pd.DataFrame()
if run_expert_ablation:
    ablation_count = min(8, len(dataset))
    ablation_batch = next(iter(make_inference_dataloader(
        Subset(dataset, list(range(ablation_count))), batch_size=ablation_count, num_workers=0
    )))
    ablation_rasters = move_to_device(ablation_batch['raster_dict'], device)
    ablation_kwargs = {
        'raster_valid_masks': move_to_device(ablation_batch.get('raster_valid_masks'), device),
        'raster_band_names': band_names,
        'meta_dict': move_to_device(ablation_batch.get('meta_dict'), device),
        'meta_valid_masks': move_to_device(ablation_batch.get('meta_valid_masks'), device),
    }
    targets, target_validity = _build_patch_targets(model, ablation_rasters, ablation_kwargs)

    def evaluate_ablation(layer_index=None, expert_index=None):
        hook = None
        if layer_index is not None:
            expert = model.encoder.layers[layer_index].moe.experts[expert_index]
            hook = expert.register_forward_hook(
                lambda module, inputs, output: torch.zeros_like(output)
            )
        try:
            with torch.inference_mode(), deterministic_routing(model):
                predictions, mask, _, _ = model(
                    raster_dict=ablation_rasters, mask_generator=fixed_mask_generator(mask_seed),
                    **ablation_kwargs,
                )
                embedding = model.extract_embedding(
                    raster_dict=ablation_rasters, token_source='pre_norm', pooling='mean_fine',
                    **ablation_kwargs,
                )
            total_loss = float(_compute_patch_loss(
                predictions, targets, target_validity, mask,
                modality_loss_weights=config['training'].get('modality_loss_weights'),
            ))
            return total_loss, reconstruction_by_modality(
                predictions, targets, target_validity, mask
            ), embedding
        finally:
            if hook is not None:
                hook.remove()

    baseline_loss, baseline_modalities, baseline_embedding = evaluate_ablation()
    selected_layers = sorted(set([0, len(model.encoder.layers) // 2, len(model.encoder.layers) - 1]))
    ablation_rows = []
    for layer_index in selected_layers:
        for expert_index in range(model.encoder.layers[layer_index].moe.num_experts):
            loss, modality_losses, embedding = evaluate_ablation(layer_index, expert_index)
            row = {
                'layer': layer_index, 'expert': expert_index,
                'reconstruction_delta': loss - baseline_loss,
                'embedding_cosine_distance': float(
                    (1 - F.cosine_similarity(baseline_embedding, embedding, dim=-1)).mean()
                ),
            }
            for modality, value in modality_losses.items():
                row[f'{modality}_loss_delta'] = value - baseline_modalities[modality]
            ablation_rows.append(row)
    ablation_summary = pd.DataFrame(ablation_rows)
    print('baseline reconstruction loss:', round(baseline_loss, 5))
    display(ablation_summary.round(5))
else:
    print('Causal ablation disabled.')

### Interpretation of the current local run

**What the test does.** A forward hook replaces one expert's output with zero while preserving router decisions, all other experts, and the exact MAE mask. Reconstruction delta measures the change from baseline masked reconstruction loss; embedding cosine distance measures the change in the downstream pre-norm mean-fine representation. Positive deltas mean the ablated expert was useful for that measurement.

**Result.** The eight-sample baseline reconstruction loss is 0.08933. All three layer-0 experts matter: their loss increases are 0.00215-0.00334 (about 2.4%-3.7% of baseline), and they produce the largest embedding changes. Their modality-wise effects differ, but none is a perfectly exclusive sensor expert.

Individual layer-7 ablations are much smaller (0.00014-0.00062), consistent with top-2 redundancy, shared projections, and residual paths. At layer 14, expert 0 is clearly more influential than its peers: its reconstruction delta is 0.00325, compared with 0.00013-0.00090 for the other experts. Combined with its larger output norm, this indicates late-layer contribution skew. It is not collapse because all experts are active and non-zero, but it should be checked on the full validation set and over checkpoint milestones.

Tiny negative per-modality deltas are numerical/sample variation and do not demonstrate that removing an expert improves the model. Eight samples are appropriate for a local causal smoke test, not a final effect estimate; report means and confidence intervals over substantially more tiles for a paper.

## 12. Checkpoint trajectory

A healthy final checkpoint can hide transient or emerging collapse. This lightweight scan uses two validation batches per milestone and reports the worst expert utilization across fused layers. It reads immutable milestone files plus `pretrained_S_best.pth`; it deliberately avoids a live `checkpoint_S.pth` that may be rewritten during training.

In [ ]:
def quick_checkpoint_health(active_model, max_batches=2):
    accumulators = {}
    with torch.inference_mode(), deterministic_routing(active_model):
        for batch_index, batch in enumerate(dataloader):
            if batch_index >= max_batches:
                break
            features = active_model.forward_features(
                raster_dict=move_to_device(batch['raster_dict'], device),
                raster_valid_masks=move_to_device(batch.get('raster_valid_masks'), device),
                raster_band_names=band_names,
                meta_dict=move_to_device(batch.get('meta_dict'), device),
                meta_valid_masks=move_to_device(batch.get('meta_valid_masks'), device),
                return_routing=True, stochastic_routing=False,
            )
            for layer, stream, routing in routing_views(features):
                update_accumulator(accumulators, ('full', 'full', layer, stream), routing)
    summary = finalize_accumulators(accumulators).query("stream == 'fused'")
    return {
        'minimum_active_route': summary['active_route_fraction'].min(),
        'minimum_gate_mass': summary['gate_mass_fraction'].min(),
        'minimum_top1': summary['top1_fraction'].min(),
        'mean_local_entropy': summary.groupby('layer')['local_entropy'].first().mean(),
        'mean_global_entropy': summary.groupby('layer')['global_entropy'].first().mean(),
    }

trajectory_summary = pd.DataFrame()
if run_checkpoint_trajectory:
    checkpoint_candidates = sorted(run_dir.glob('checkpoint_S_epoch_*.pth'))
    if checkpoint_path.exists():
        checkpoint_candidates.append(checkpoint_path)
    trajectory_rows = []
    for path in checkpoint_candidates:
        active_model = build_model_from_dataset(config, dataset, str(path), device)
        trajectory_rows.append({'checkpoint': path.name, **quick_checkpoint_health(active_model)})
        del active_model
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    trajectory_summary = pd.DataFrame(trajectory_rows)
    display(trajectory_summary.round(4))
else:
    print('Checkpoint trajectory disabled.')

### Interpretation of the current local run

**What the test does.** It loads each immutable milestone checkpoint and evaluates two fixed validation batches. The minimum active fraction and gate mass are conservative collapse indicators; entropy summarizes router uncertainty. A trajectory is more informative than one final checkpoint because starvation can emerge or disappear during training.

**Current status.** This test is disabled in local fixture mode, so the message `Checkpoint trajectory disabled` is expected and contains no result. Enable it on the VM where milestone checkpoints are available. It is a routing-health test only: it cannot establish semantic or functional specialization, and two batches are intended as a fast warning signal rather than a final estimate.

## 13. Interpretation checklist

### Overall conclusion for this local fixture

The checkpoint does **not** show the previous failure mode. Experts are all executed, have non-zero gate mass and non-zero functions, and routing is not meaningfully tied to absolute patch position. Routing changes with sensor availability and is associated with Dynamic World land-cover classes. Expert outputs become more differentiated through depth, and causal ablation confirms that experts contribute unequally.

Two qualifications remain. First, learned fusion is valid and non-collapsed but varies only modestly across patches, so it currently behaves more like a sensor weighting prior than a strongly local quality selector. Second, deep layer 14 gives expert 0 a larger output norm and ablation effect than the other experts. This is contribution skew, not collapse, but it should be monitored over the full validation set and checkpoint trajectory.

Therefore the local result is **healthy enough to continue evaluation**, not sufficient by itself for a paper claim. Final evidence requires the full validation split, milestone trajectories, multiple seeds, tile-aware uncertainty, and downstream performance.

Interpret the outputs in this order:

1. **Health:** no expert should remain near zero in active routes or gate mass across many layers/checkpoints. Compare full and masked inputs.
2. **Position:** position NMI and same-position agreement should be close to their permutation/chance baselines. Raw values without baselines are not interpretable.
3. **Modality:** layer-0 differences can indicate modality-conditioned routing. Later sensor-removal differences show sensitivity, not automatically dedicated modality experts.
4. **Functions:** output cosine substantially below one and non-zero causal effects show that experts are not identical or dead.
5. **Semantics:** class enrichment and expert/class NMI above the permutation null support semantic association. Confirm patterns across checkpoints/seeds before claiming specialization.
6. **Visuals:** compare several explicit dataset indices. Do not infer specialization from one attractive routing map.

There is no universal pass/fail threshold. The strongest evidence combines healthy execution, low positional shortcut dependence, reproducibility, semantic enrichment, and distinct causal effects.